# 07 One-Click LexAI Runner (Final Stable)

Run cells top-to-bottom once.

Design goals:
- No hard failures across cells (graceful skip if prerequisite is missing)
- Databricks workspace notebook path handling via adapter fallback
- Correct browser links (driver-proxy), not localhost links
- Optional non-blocking Streamlit startup


In [0]:
# CELL 1: Runtime Flags
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = False

# Only use this if import stack is broken and you accept manual restart+rerun.
FORCE_REPAIR_IMPORT_STACK = False

FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

REPO_DIR_OVERRIDE = "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform"

SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

# Shared runtime flags
REPO_OK = False
DEPS_OK = False
ENGINE_READY = False
API_READY = False
STREAMLIT_READY = False

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FORCE_REPAIR_IMPORT_STACK": FORCE_REPAIR_IMPORT_STACK,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
    "REPO_DIR_OVERRIDE": REPO_DIR_OVERRIDE,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': False, 'FORCE_REPAIR_IMPORT_STACK': False, 'FASTAPI_PORT': 8000, 'STREAMLIT_PORT': 8501, 'REPO_DIR_OVERRIDE': '/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform'}


In [0]:
# CELL 2: Resolve repo path safely
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_required_files(repo_dir: Path) -> bool:
    return (
        (repo_dir / "apps" / "fastapi_app.py").exists()
        and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()
    )


def _safe_walk_for_repo(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _ in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        p = Path(dirpath)
        try:
            if _repo_has_required_files(p):
                return p
        except Exception:
            continue
    return None


def _context_repo_guess():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()  # /Users/<email>/<repo>/notebooks/...
        if not nb_path:
            return None
        pp = Path(nb_path)
        ws_repo = Path("/Workspace") / Path(*pp.parent.parent.parts[1:])
        if ws_repo.exists() and _repo_has_required_files(ws_repo):
            return ws_repo
    except Exception:
        pass
    return None


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_required_files(p):
            return p

    cwd = Path(os.getcwd()).resolve()
    for cand in [cwd] + list(cwd.parents):
        try:
            if _repo_has_required_files(cand):
                return cand
        except Exception:
            continue

    g = _context_repo_guess()
    if g is not None:
        return g

    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hit = _safe_walk_for_repo(root)
        if hit is not None:
            return hit

    return Path(os.getcwd()).resolve()


REPO_DIR = resolve_repo_dir()
if _repo_has_required_files(REPO_DIR):
    REPO_OK = True
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    log(f"Repo root: {REPO_DIR}")
    print("[CELL 2] REPO_OK=True")
else:
    REPO_OK = False
    print("[CELL 2] REPO_OK=False; unresolved repo path:", REPO_DIR)
    print("[CELL 2] Set REPO_DIR_OVERRIDE to your repo path and rerun from Cell 1.")


[05:43:34] Repo root: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 2] REPO_OK=True


In [0]:
# CELL 3: Dependency preflight + import sanity (no forced restart)
import importlib
import importlib.metadata as ilm
import subprocess
import sys
from pathlib import Path

DEPS_OK = False

if not REPO_OK:
    print("[CELL 3] Skipped: REPO_OK=False")
else:
    req_file = Path("apps/requirements.txt")
    if not req_file.exists():
        print(f"[CELL 3] Missing requirements file: {req_file}")
    else:
        required_specs = [
            "fastapi",
            "uvicorn",
            "streamlit",
            "requests",
            "pydantic",
            "sentence-transformers",
            "transformers>=4.30.0",
            "accelerate>=0.20.0",
            "mlflow",
            "databricks-sdk",
            "typing_extensions>=4.6.0",
        ]

        def _pkg_name(spec: str) -> str:
            for sep in [">=", "==", "<=", "~=", ">", "<"]:
                if sep in spec:
                    return spec.split(sep)[0].strip()
            return spec.strip()

        missing_specs = []
        for spec in required_specs:
            try:
                ilm.version(_pkg_name(spec))
            except Exception:
                missing_specs.append(spec)

        print("[CELL 3] Missing specs:", missing_specs)

        if missing_specs and AUTO_INSTALL_MISSING:
            cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)] + missing_specs
            try:
                print("[CELL 3] Installing missing specs...")
                subprocess.check_call(cmd)
            except Exception as e:
                print("[CELL 3] Package installation failed:", e)
        elif missing_specs and not AUTO_INSTALL_MISSING:
            print("[CELL 3] AUTO_INSTALL_MISSING=False and packages missing.")

        # Clear stale import cache for known circular-import offenders.
        for k in list(sys.modules.keys()):
            if k.startswith(("accelerate", "transformers")):
                del sys.modules[k]

        try:
            import typing_extensions
            importlib.reload(typing_extensions)
            _ = typing_extensions.TypeIs

            import transformers  # noqa:F401
            import accelerate  # noqa:F401
            DEPS_OK = True
            print("[CELL 3] DEPS_OK=True")
        except Exception as e:
            DEPS_OK = False
            print("[CELL 3] Import sanity failed:", e)
            if FORCE_REPAIR_IMPORT_STACK:
                print("[CELL 3] FORCE_REPAIR_IMPORT_STACK=True -> running repair install")
                repair_cmd = [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "--upgrade",
                    "typing_extensions>=4.6.0",
                    "accelerate>=0.20.0",
                    "transformers>=4.30.0",
                ]
                try:
                    subprocess.check_call(repair_cmd)
                    print("[CELL 3] Repair install done.")
                    print("[CELL 3] Manual step required: run dbutils.library.restartPython() then rerun from Cell 1.")
                except Exception as e2:
                    print("[CELL 3] Repair install failed:", e2)


[CELL 3] Missing specs: []
[CELL 3] Import sanity failed: module 'numpy' has no attribute 'dtypes'


In [0]:
# CELL 4: Spark context and browser-link base
from pyspark.sql import SparkSession

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"
compute_mode = "unknown"
DRIVER_PROXY_BASE = ""
DRIVER_PROXY_SUPPORTED = False

try:
    spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
    print("[CELL 4] Spark session ready:", bool(spark))
except Exception as e:
    spark = None
    print("[CELL 4] Spark unavailable:", e)


def _safe_conf(key: str, default: str = ""):
    if spark is None:
        return default
    try:
        v = spark.conf.get(key)
        if v and str(v).strip():
            return str(v).strip()
    except Exception:
        pass
    return default


def _opt_to_str(opt):
    try:
        if hasattr(opt, "isDefined") and opt.isDefined():
            return str(opt.get())
    except Exception:
        pass
    try:
        return str(opt.get())
    except Exception:
        pass
    return ""


def _ctx_tag(ctx, key: str):
    try:
        tags = ctx.tags()
        return _opt_to_str(tags.get(key))
    except Exception:
        pass
    try:
        tags = ctx.tags()
        return str(tags.apply(key))
    except Exception:
        pass
    return ""


ctx = None
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
except Exception:
    pass

cluster_id = _safe_conf("spark.databricks.clusterUsageTags.clusterId", "") or (_ctx_tag(ctx, "clusterId") if ctx else "")
org_id = _safe_conf("spark.databricks.clusterUsageTags.orgId", "") or (_ctx_tag(ctx, "orgId") if ctx else "")
compute_mode = _safe_conf("spark.databricks.clusterUsageTags.clusterSource", "") or (_ctx_tag(ctx, "clusterSource") if ctx else "") or "unknown"

ws_from_conf = _safe_conf("spark.databricks.workspaceUrl", "")
ws_from_ctx = _opt_to_str(ctx.browserHostName()) if ctx else ""
ws_api = _opt_to_str(ctx.apiUrl()) if ctx else ""
workspace_url = ws_from_conf or ws_from_ctx
if (not workspace_url) and ws_api:
    workspace_url = ws_api.replace("https://", "").split("/")[0]

if not cluster_id:
    cluster_id = "unknown"
if not org_id:
    org_id = "unknown"
if not workspace_url:
    workspace_url = "unknown"

if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
    DRIVER_PROXY_BASE = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}"
    DRIVER_PROXY_SUPPORTED = True

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)
print("[CELL 4] compute_mode:", compute_mode)
print("[CELL 4] DRIVER_PROXY_SUPPORTED:", DRIVER_PROXY_SUPPORTED)
print("[CELL 4] DRIVER_PROXY_BASE:", DRIVER_PROXY_BASE or "unavailable")


[CELL 4] Spark session ready: True
[CELL 4] cluster_id: 0302-051116-1ns9zuy8-v2n
[CELL 4] org_id: unknown
[CELL 4] workspace_url: dbc-afb2e98d-d930.cloud.databricks.com
[CELL 4] compute_mode: unknown
[CELL 4] DRIVER_PROXY_SUPPORTED: False
[CELL 4] DRIVER_PROXY_BASE: unavailable


In [0]:
# CELL 4.5: Optional repair marker
if FORCE_REPAIR_IMPORT_STACK and not DEPS_OK:
    print("[CELL 4.5] Repair mode active and DEPS_OK=False.")
    print("[CELL 4.5] If you ran repair installs, run dbutils.library.restartPython() and rerun from Cell 1.")
else:
    print("[CELL 4.5] No repair action needed.")


[CELL 4.5] No repair action needed.


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter
from pathlib import Path
import importlib
import os

ENGINE_READY = False
status = {}

if not REPO_OK:
    print("[CELL 5] Skipped: REPO_OK=False")
elif not DEPS_OK:
    print("[CELL 5] Skipped: DEPS_OK=False (fix Cell 3 first)")
else:
    import apps.lexai06_notebook_adapter as _adapter
    importlib.reload(_adapter)
    NotebookEngine = _adapter.NotebookEngine

    workspace_candidates = [
        "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
        "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
        "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
        "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
        str(Path(REPO_DIR) / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"),
        str(Path(REPO_DIR) / "apps" / "notebook_06_snapshot.ipynb"),
    ]

    print("[CELL 5] Notebook candidates:")
    for c in workspace_candidates:
        try:
            print(" -", c, "exists=", Path(c).exists())
        except Exception:
            print(" -", c, "exists=ERROR")

    last_err = None
    for cand in workspace_candidates:
        try:
            os.environ["LEXAI06_NOTEBOOK_PATH"] = cand
            engine = NotebookEngine(notebook_path=Path(cand))
            status = engine.initialize()
            ENGINE_READY = bool(status.get("ready"))
            print(f"[CELL 5] Initialized using candidate: {cand}")
            break
        except Exception as e:
            last_err = e
            print(f"[CELL 5] Candidate failed: {cand} -> {e}")

    if not ENGINE_READY:
        print("[CELL 5] Engine not ready. Last error:", last_err)
    else:
        print("[CELL 5] Engine initialized")
        for k, v in status.items():
            print(f"  - {k}: {v}")


[CELL 5] Skipped: DEPS_OK=False (fix Cell 3 first)


In [0]:
# CELL 6: Smoke test
if RUN_SMOKE_TEST and ENGINE_READY:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] Skipped (RUN_SMOKE_TEST or ENGINE_READY condition not met).")


[CELL 6] Skipped (RUN_SMOKE_TEST or ENGINE_READY condition not met).


In [0]:
# CELL 7: Start FastAPI and print exact browser links
import threading
import time
import requests
import uvicorn

API_READY = False
API_BROWSER_HEALTH_URL = ""
API_BROWSER_DOCS_URL = ""

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")


def _wait_api_local(port: int, timeout_sec: int = 60):
    start = time.time()
    url = f"http://127.0.0.1:{port}/health"
    while (time.time() - start) < timeout_sec:
        try:
            r = requests.get(url, timeout=2)
            if r.status_code == 200:
                return True, r.json()
        except Exception:
            pass
        time.sleep(1)
    return False, {}


if START_FASTAPI and ENGINE_READY:
    if FASTAPI_THREAD is None or not FASTAPI_THREAD.is_alive():
        from apps.fastapi_app import app
        selected_port = int(FASTAPI_PORT)
        started = False
        for p in [selected_port, selected_port + 1, selected_port + 2]:
            try:
                config = uvicorn.Config(app, host="0.0.0.0", port=int(p), log_level="info")
                FASTAPI_SERVER = uvicorn.Server(config)
                FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
                FASTAPI_THREAD.start()
                ok, _ = _wait_api_local(int(p), timeout_sec=20)
                if ok:
                    FASTAPI_PORT = int(p)
                    started = True
                    break
            except Exception:
                continue

        if started:
            globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
            globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
            print(f"[CELL 7] FastAPI started on port {FASTAPI_PORT}")
        else:
            print("[CELL 7] FastAPI failed to start on tried ports.")
    else:
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")

    ok, body = _wait_api_local(int(FASTAPI_PORT), timeout_sec=60)
    API_READY = ok
    print("[CELL 7] Local health check:", "OK" if ok else "FAILED")
    if body:
        print(body)

    print("[CELL 7] Local-only URL (do not open in browser tab):", f"http://127.0.0.1:{FASTAPI_PORT}/health")

    if DRIVER_PROXY_SUPPORTED:
        API_BROWSER_HEALTH_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/health"
        API_BROWSER_DOCS_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/docs"
        print("[CELL 7] Browser URL (health):", API_BROWSER_HEALTH_URL)
        print("[CELL 7] Browser URL (docs):", API_BROWSER_DOCS_URL)
        try:
            displayHTML(f'<a href="{API_BROWSER_DOCS_URL}" target="_blank">Open FastAPI Docs</a>')
        except Exception:
            pass
    else:
        print("[CELL 7] Driver proxy unsupported in this compute context (browser tabs will not work).")
else:
    print("[CELL 7] Skipped (START_FASTAPI or ENGINE_READY condition not met).")


[CELL 7] Skipped (START_FASTAPI or ENGINE_READY condition not met).


In [0]:
# CELL 8: FastAPI smoke call
import requests

if API_READY:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] local /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] local /v1/legal/answer status:", r.status_code)
        try:
            body = r.json()
            print("[CELL 8] answer preview:", body.get("answer", "")[:500])
        except Exception:
            print("[CELL 8] raw response:", r.text[:500])

        if API_BROWSER_DOCS_URL:
            print("[CELL 8] Browser docs URL:", API_BROWSER_DOCS_URL)
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] Skipped: API_READY=False")


[CELL 8] Skipped: API_READY=False


In [0]:
# CELL 9: Optional Streamlit start (non-blocking) and browser URL
import subprocess

STREAMLIT_READY = False
STREAMLIT_BROWSER_URL = ""
STREAMLIT_PROC = globals().get("STREAMLIT_PROC")

if START_STREAMLIT and API_READY:
    if STREAMLIT_PROC is not None and STREAMLIT_PROC.poll() is None:
        STREAMLIT_READY = True
        print(f"[CELL 9] Streamlit already running on port {STREAMLIT_PORT}")
    else:
        selected_port = int(STREAMLIT_PORT)
        started = False
        for p in [selected_port, selected_port + 1, selected_port + 2]:
            try:
                cmd = [
                    sys.executable,
                    "-m",
                    "streamlit",
                    "run",
                    "apps/streamlit_app.py",
                    "--server.port", str(p),
                    "--server.address", "0.0.0.0",
                ]
                env = os.environ.copy()
                env["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                STREAMLIT_PROC = proc
                STREAMLIT_PORT = int(p)
                STREAMLIT_READY = True
                globals()["STREAMLIT_PROC"] = STREAMLIT_PROC
                started = True
                print(f"[CELL 9] Streamlit started on port {STREAMLIT_PORT}")
                break
            except Exception:
                continue

        if not started:
            print("[CELL 9] Streamlit failed to start on tried ports.")

    if STREAMLIT_READY and DRIVER_PROXY_SUPPORTED:
        STREAMLIT_BROWSER_URL = f"{DRIVER_PROXY_BASE}/{STREAMLIT_PORT}/"
        print("[CELL 9] Browser Streamlit URL:", STREAMLIT_BROWSER_URL)
        try:
            displayHTML(f'<a href="{STREAMLIT_BROWSER_URL}" target="_blank">Open Streamlit UI</a>')
        except Exception:
            pass
    elif STREAMLIT_READY:
        print("[CELL 9] Streamlit running, but driver proxy unsupported for browser access in this context.")
else:
    print("[CELL 9] Skipped (START_STREAMLIT or API_READY condition not met).")


[CELL 9] Skipped (START_STREAMLIT or API_READY condition not met).


In [0]:
# CELL 10: Stop helper
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")

if "STREAMLIT_PROC" in globals() and globals().get("STREAMLIT_PROC") is not None:
    proc = globals()["STREAMLIT_PROC"]
    try:
        if proc.poll() is None:
            proc.terminate()
            print("[CELL 10] Streamlit process termination requested")
        else:
            print("[CELL 10] Streamlit process already stopped")
    except Exception as e:
        print("[CELL 10] Streamlit stop failed:", e)
else:
    print("[CELL 10] Streamlit was not running")


[CELL 10] FastAPI was not running
[CELL 10] Streamlit was not running
